# Qwen3-0.6B LoRA 指令微调（干净重建版）

这份 notebook 使用本地缓存的 `Qwen/Qwen3-0.6B`，在本地 `alpaca_zh_10k.json` 数据集上做 LoRA 指令微调（SFT），并保留一个 smoke test 配置，方便先验证整条训练链路是否正常。

这一版额外加入了 `conda normal` 环境兼容补丁：会先清理用户目录中的 `site-packages` 干扰，并关闭当前环境里有冲突的 AWQ/TensorFlow 导入路径，目标是让 notebook 至少能稳定跑完最小训练。 

## 1. 安装依赖

这一部分先安装 LoRA 微调需要的基础库。`transformers` 负责模型加载与训练，`peft` 负责注入低秩适配器，`datasets` 负责处理训练数据。

LoRA 的核心思想是把全参数更新写成低秩增量：

$$
W' = W + \Delta W, \qquad \Delta W = BA, \qquad \mathrm{rank}(\Delta W)=r \ll d
$$

这样训练时只需要更新很少的新增参数，而不是整块权重矩阵。

In [ ]:
%pip install -q transformers torch safetensors peft datasets accelerate sentencepiece

## 2. 定位本地模型和数据

这里把本地模型目录、数据集路径和训练输出目录定义清楚。形式上，我们要准备的是一个监督学习数据集：

$$
\mathcal{D} = \{(x_i, y_i)\}_{i=1}^{N}
$$

其中 $x_i$ 表示指令输入，$y_i$ 表示参考答案。后续我们会把它们转成模型可以直接消费的 token 序列。

In [3]:
from pathlib import Path
import json
import os
import sys

os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("TRANSFORMERS_NO_TF", "1")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")

removed_user_site = [p for p in list(sys.path) if "AppData\\Roaming\\Python" in p or ".local/lib/python" in p]
sys.path = [p for p in sys.path if p not in removed_user_site]
print("Removed user site paths =", removed_user_site)

import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, set_seed
from peft import LoraConfig, TaskType, get_peft_model
import peft.import_utils as peft_import_utils
import peft.tuners.lora.awq as peft_awq

BASE_MODEL_DIR = Path.home() / ".cache" / "huggingface" / "hub" / "models--Qwen--Qwen3-0.6B" / "snapshots" / "c1899de289a04d12100db370d81485cdf75e47ca"
DATA_PATH = Path.home() / "Desktop" / "0414组会" / "alpaca_zh_10k.json"
OUTPUT_DIR = Path.cwd() / "qwen3_lora_alpaca_zh_10k_clean_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

required_model_files = ["config.json", "model.safetensors", "tokenizer.json"]
print("BASE_MODEL_DIR =", BASE_MODEL_DIR)
print("DATA_PATH =", DATA_PATH)
print("OUTPUT_DIR =", OUTPUT_DIR)
print("model exists =", BASE_MODEL_DIR.exists())
print("data exists =", DATA_PATH.exists())
for name in required_model_files:
    print(f"{name}:", (BASE_MODEL_DIR / name).exists())

Removed user site paths = ['C:\\Users\\z5364\\AppData\\Roaming\\Python\\Python310\\site-packages']
BASE_MODEL_DIR = C:\Users\z5364\.cache\huggingface\hub\models--Qwen--Qwen3-0.6B\snapshots\c1899de289a04d12100db370d81485cdf75e47ca
DATA_PATH = C:\Users\z5364\Desktop\0414组会\alpaca_zh_10k.json
OUTPUT_DIR = c:\Users\z5364\Desktop\0414组会\0414组会\qwen3_lora_alpaca_zh_10k_clean_output
model exists = True
data exists = True
config.json: True
model.safetensors: True
tokenizer.json: True


## 3. 加载基座模型并注入 LoRA

这一部分先固定随机种子，再根据当前硬件自动选择 `cuda` 或 `cpu`。如果 GPU 可用，就优先使用 GPU 推理和训练。

对一个线性层来说，原始映射是：

$$
h = xW^{\top}
$$

LoRA 会把它改写成：

$$
h = x(W + BA)^{\top}
$$

其中基座参数保持冻结，只训练低秩矩阵 $A$ 和 $B$。

In [4]:
set_seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE == "cuda" and torch.cuda.is_bf16_supported() else (torch.float16 if DEVICE == "cuda" else torch.float32)

torch.set_num_threads(max(1, (os.cpu_count() or 1) // 2))
if DEVICE == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True

tokenizer = AutoTokenizer.from_pretrained(str(BASE_MODEL_DIR), trust_remote_code=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    str(BASE_MODEL_DIR),
    dtype=DTYPE,
    trust_remote_code=False,
)
base_model.to(DEVICE)
base_model.config.use_cache = False
if hasattr(base_model, "gradient_checkpointing_enable"):
    base_model.gradient_checkpointing_enable()
if hasattr(base_model, "enable_input_require_grads"):
    base_model.enable_input_require_grads()

desired_targets = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
linear_suffixes = sorted({name.split(".")[-1] for name, module in base_model.named_modules() if isinstance(module, torch.nn.Linear)})
target_modules = [name for name in desired_targets if name in linear_suffixes]
if not target_modules:
    raise RuntimeError("Could not find any LoRA target modules in the model.")

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=target_modules,
)

# `normal` 环境里如果装了不兼容的 autoawq，这里会误触发 AWQ 分支。
# 对普通的 Qwen 线性层 LoRA 来说，直接禁用该分支更稳妥。
peft_import_utils.is_auto_awq_available = lambda: False
peft_awq.is_auto_awq_available = lambda: False

model = get_peft_model(base_model, lora_config)
model.config.use_cache = False
print(f"Loaded model on {DEVICE} with dtype={DTYPE}")
print("LoRA target modules =", target_modules)
model.print_trainable_parameters()

Loaded model on cuda with dtype=torch.bfloat16
LoRA target modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
trainable params: 5,046,272 || all params: 601,096,192 || trainable%: 0.8395


## 4. 构造指令数据与监督目标

这里把 `alpaca_zh_10k.json` 中的 `instruction / input / output` 组装成对话式 prompt，并把答案部分作为监督目标。

标准的指令微调损失可以写成：

$$
\mathcal{L}_{\mathrm{SFT}} = - \sum_{t \in y} \log p_{\phi}(y_t \mid x, y_{<t})
$$

实现时会把 prompt 部分的 label 置为 `-100`，这样 loss 只在答案 token 上计算。

In [5]:
SYSTEM_PROMPT = "你是一个乐于助人的中文助手，请根据用户的指令给出准确、简洁、清晰的回答。"
MAX_LENGTH = 256
MAX_ANSWER_LENGTH = 128
SMOKE_TRAIN_SAMPLES = 64
SMOKE_EVAL_SAMPLES = 8

def load_alpaca_records(path: Path):
    records = json.loads(path.read_text(encoding="utf-8"), strict=False)
    if not isinstance(records, list):
        raise TypeError(f"Expected a JSON list, got {type(records).__name__}")

    cleaned = []
    for row in records:
        if not isinstance(row, dict):
            continue
        instruction = str(row.get("instruction", "")).strip()
        user_input = str(row.get("input", "")).strip()
        output = str(row.get("output", "")).strip()
        if not instruction or not output:
            continue
        cleaned.append({
            "instruction": instruction,
            "input": user_input,
            "output": output,
        })
    return cleaned

def build_user_prompt(row):
    instruction = row["instruction"].strip()
    user_input = row["input"].strip()
    parts = [f"任务指令：\n{instruction}"]
    if user_input:
        parts.append(f"补充输入：\n{user_input}")
    parts.append("请直接给出回答。")
    return "\n\n".join(parts)

def build_messages(row):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(row)},
    ]

def build_chat_prompt(row):
    return tokenizer.apply_chat_template(
        build_messages(row),
        add_generation_prompt=True,
        enable_thinking=False,
        tokenize=False,
    )

def tokenize_row(row):
    prompt_text = build_chat_prompt(row)
    answer_text = row["output"].strip() + (tokenizer.eos_token or "")

    prompt_ids = tokenizer(prompt_text, add_special_tokens=False).input_ids
    answer_ids = tokenizer(answer_text, add_special_tokens=False).input_ids[:MAX_ANSWER_LENGTH]

    max_prompt_length = max(0, MAX_LENGTH - len(answer_ids))
    prompt_ids = prompt_ids[:max_prompt_length]

    input_ids = prompt_ids + answer_ids
    labels = [-100] * len(prompt_ids) + answer_ids

    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": labels,
        "length": len(input_ids),
    }

records = load_alpaca_records(DATA_PATH)
if not records:
    raise ValueError("No valid records found in alpaca_zh_10k.json")

dataset = Dataset.from_list(records).shuffle(seed=42)

train_ds = dataset.select(range(min(SMOKE_TRAIN_SAMPLES, len(dataset))))
remaining = dataset.select(range(min(SMOKE_TRAIN_SAMPLES, len(dataset)), len(dataset)))
if len(remaining):
    eval_ds = remaining.select(range(min(SMOKE_EVAL_SAMPLES, len(remaining))))
else:
    eval_ds = dataset.select(range(min(SMOKE_EVAL_SAMPLES, len(dataset))))

train_tok = train_ds.map(tokenize_row, remove_columns=train_ds.column_names)
eval_tok = eval_ds.map(tokenize_row, remove_columns=eval_ds.column_names)

sample_row = records[0]
print("records =", len(records))
print("train size =", len(train_ds))
print("eval size =", len(eval_ds))
print("sample record =", sample_row)
print("sample prompt =\n", build_chat_prompt(sample_row))
print("sample output =", sample_row["output"])
print("sample tokenized length =", train_tok[0]["length"])


Map:   0%|          | 0/64 [00:00<?, ? examples/s]

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

records = 10000
train size = 64
eval size = 8
sample prompt =
 <|im_start|>system
你是一个乐于助人的中文助手，请根据用户的指令给出准确、简洁、清晰的回答。<|im_end|>
<|im_start|>user
### Instruction:
保持健康的三个提示。<|im_end|>
<|im_start|>assistant
<think>

</think>




## 5. Padding 与训练超参数

不同样本的长度不一致，所以需要在 `collate_fn` 里补齐成同一个 batch。可以把一个 batch 的目标理解成样本损失的平均：

$$
\mathcal{L}_{\mathrm{batch}} = \frac{1}{B} \sum_{i=1}^{B} \mathcal{L}_{\mathrm{SFT}}^{(i)}
$$

这里为了 smoke test，把训练步数设得很小，目的是先确认整条训练链路能够顺利跑通。

In [6]:
def collate_fn(features):
    input_ids = [torch.tensor(item["input_ids"], dtype=torch.long) for item in features]
    attention_mask = [torch.tensor(item["attention_mask"], dtype=torch.long) for item in features]
    labels = [torch.tensor(item["labels"], dtype=torch.long) for item in features]

    return {
        "input_ids": torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id),
        "attention_mask": torch.nn.utils.rnn.pad_sequence(attention_mask, batch_first=True, padding_value=0),
        "labels": torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=-100),
    }

MAX_STEPS = 5

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    overwrite_output_dir=True,
    do_train=True,
    do_eval=True,
    eval_strategy="steps",
    eval_steps=5,
    save_strategy="steps",
    save_steps=5,
    save_total_limit=2,
    logging_strategy="steps",
    logging_steps=1,
    learning_rate=2e-4,
    weight_decay=0.0,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=1,
    num_train_epochs=1,
    max_steps=MAX_STEPS,
    bf16=DEVICE == "cuda" and torch.cuda.is_bf16_supported(),
    fp16=DEVICE == "cuda" and not torch.cuda.is_bf16_supported(),
    tf32=DEVICE == "cuda",
    optim="adamw_torch",
    report_to="none",
    remove_unused_columns=False,
    group_by_length=True,
    length_column_name="length",
    dataloader_num_workers=0,
    dataloader_pin_memory=DEVICE == "cuda",
    run_name="qwen3-lora-alpaca-zh-10k-clean",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    data_collator=collate_fn,
)

print("Trainer ready")

Trainer ready


## 6. 开始训练 LoRA 参数

这一步是真正的优化过程。基座模型参数 $\theta$ 保持冻结，只更新 LoRA 中的低秩参数 $\phi$，因此优化目标可以写成：

$$
\min_{\phi} \; \mathcal{L}_{\mathrm{SFT}}(\theta, \phi)
$$

因为这里是 smoke test，只跑很少的 step，用来快速验证训练循环、显存配置和保存流程是否正常。

In [ ]:
train_result = trainer.train()
print(train_result)

## 7. 保存 Adapter、合并权重并做推理

训练结束后，LoRA 一般有两种用法：一种是单独保存 adapter，另一种是把低秩增量合并回基座权重。合并后的形式是：

$$
W_{\mathrm{merge}} = W + BA
$$

这样部署时就不必额外加载 adapter 了。最后再做一次生成测试，确认微调后的模型可以正常回答。

In [ ]:
ADAPTER_DIR = OUTPUT_DIR / "adapter"
MERGED_DIR = OUTPUT_DIR / "merged"
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
MERGED_DIR.mkdir(parents=True, exist_ok=True)

trainer.model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
print("Saved LoRA adapter to", ADAPTER_DIR)

merged_model = trainer.model.merge_and_unload()
merged_model.to(DEVICE)
merged_model.save_pretrained(str(MERGED_DIR), safe_serialization=True)
tokenizer.save_pretrained(str(MERGED_DIR))
print("Saved merged model to", MERGED_DIR)

def chat(model_obj, prompt: str, system: str = SYSTEM_PROMPT, max_new_tokens: int = 128) -> str:
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        enable_thinking=False,
        return_dict=True,
        return_tensors="pt",
    )
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = model_obj.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print(chat(merged_model, "请用一句话解释 LoRA 微调的核心思想。"))